# 2.1 - Feature Engineering: Temporal & Lag Features

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Generar features temporales y de lags para el dataset de commodities:

1. **Temporal Features:** Extraer componentes temporales (year, month, quarter, day_of_week, etc.)
2. **Lag Features:** Crear variables rezagadas [1, 7, 30 días] para precios y predictores

Este notebook es parte del pipeline modular de feature engineering (Fase 2.0).

## Setup

In [51]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import (
    INTERIM_COMMODITIES_DIR, 
    INTERIM_PREDICTORS_DIR, 
    PROCESSED_DIR, 
    START_DATE,
    END_DATE,
    logger
)

# Definir lista de commodities agrícolas (TARGETS para predicción)
AGRICULTURAL_COMMODITIES = [
    'Corn', 'Soybeans', 'Wheat', 'Wheat_Kansas', 'Oat',
    'Soybean_Meal', 'Soybean_Oil', 'Sugar', 'Coffee', 'Cocoa',
    'Cotton', 'Lumber', 'Live_Cattle', 'Feeder_Cattle', 'Lean_Hogs'
]

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Interim commodities: {INTERIM_COMMODITIES_DIR}")
print(f"✓ Interim predictors: {INTERIM_PREDICTORS_DIR}")
print(f"✓ Processed output: {PROCESSED_DIR}")
print(f"\n✓ Target commodities (agricultural): {len(AGRICULTURAL_COMMODITIES)}")
print(f"  {', '.join(AGRICULTURAL_COMMODITIES[:5])}...")
print(f"\n✓ Período de análisis: {START_DATE} → {END_DATE}")

✓ Base directory: c:\Users\AdministradorIT\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Interim commodities: C:\Users\AdministradorIT\OneDrive\UBA\Taller de Programacion\TPFinal\data\interim\commodities
✓ Interim predictors: C:\Users\AdministradorIT\OneDrive\UBA\Taller de Programacion\TPFinal\data\interim\predictors
✓ Processed output: C:\Users\AdministradorIT\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed

✓ Target commodities (agricultural): 15
  Corn, Soybeans, Wheat, Wheat_Kansas, Oat...

✓ Período de análisis: 2000-01-01 → 2025-11-10


## 1. Cargar Datos Base

Cargamos dataset consolidado desde `data/processed/commodities_base_consolidated.csv`.

**IMPORTANTE:** Este dataset YA INCLUYE variables climáticas (ONI, Temp_Global_Grain, Precip_Global_Grain, GDD, ET0, Heat_Stress_Days, etc.) generadas por `src/data/process.py`.

In [52]:
# Cargar dataset consolidado (commodities + predictores + clima)
input_file = PROCESSED_DIR / 'commodities_base_consolidated.csv'

if not input_file.exists():
    raise FileNotFoundError(f"No se encontró {input_file}. Ejecuta src/data/process.py primero.")

df = pd.read_csv(input_file, parse_dates=['date'])

print(f"✓ Dataset consolidado cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")
print(f"  Período: {df['date'].min().date()} → {df['date'].max().date()}")

# Identificar variables climáticas
climate_cols = [c for c in df.columns if any(x in c for x in ['ONI', 'Temp_', 'Precip_', 'GDD_', 'ET0_', 'Heat_Stress', 'RH_', 'Solar', 'Wind'])]
print(f"\n✓ Variables climáticas incluidas: {len(climate_cols)}")
if climate_cols:
    print(f"  {', '.join(sorted(climate_cols))}")

display(df.head())

✓ Dataset consolidado cargado: commodities_base_consolidated.csv
  Dimensiones: (6981, 99)
  Período: 1999-01-04 → 2025-11-10

✓ Variables climáticas incluidas: 10
  ET0_Global_Grain, GDD_Global_Grain, Heat_Stress_Days, ONI, Precip_Deficit, Precip_Global_Grain, RH_Global_Grain, SolarRad_Global_Grain, Temp_Global_Grain, WindSpeed_Global_Grain


,date,Baltic_Dry_Index,Brent_Crude,Cocoa,Coffee,Copper,Corn,Cotton,Crude_Oil,Ethanol,...,psd_united_states_Ending_Stocks,psd_united_states_Exports,psd_united_states_Stock_to_Use_Ratio,psd_argentina_Production,psd_argentina_Exports,psd_argentina_Stock_to_Use_Ratio,psd_china_Imports,psd_china_Crush,psd_china_Ending_Stocks,psd_china_Stock_to_Use_Ratio
0,1999-01-04,784.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1999-01-05,779.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1999-01-06,791.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1999-01-07,792.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1999-01-08,791.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Convertir a Formato Ancho

Transformamos de formato largo (1 fila por commodity-fecha) a formato ancho (1 fila por fecha, 1 columna por commodity).

## 2. Filtrar por Período de Análisis

Filtramos datos desde 2000 en adelante para garantizar calidad y relevancia.

In [53]:
# Filtrar por período de análisis (>= 2000)
df = df[df['date'] >= pd.to_datetime(START_DATE)].reset_index(drop=True)

print(f"✓ Dataset filtrado (>= {START_DATE}): {df.shape}")
print(f"  Período: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Columnas: {len(df.columns)} (1 date + {len(df.columns)-1} features base)")

# Identificar tipos de variables
agricultural_vars = [c for c in df.columns if any(ag in c for ag in AGRICULTURAL_COMMODITIES) and not c.endswith('_volume') and c != 'date']
volume_vars = [c for c in df.columns if c.endswith('_volume')]
climate_vars = [c for c in df.columns if any(x in c for x in ['ONI', 'Temp_', 'Precip_', 'GDD_', 'ET0_', 'Heat_Stress', 'RH_', 'Solar', 'Wind'])]
predictor_vars = [c for c in df.columns if c not in agricultural_vars and c not in volume_vars and c not in climate_vars and c != 'date']

print(f"\nComposición del dataset:")
print(f"  Targets agrícolas: {len(agricultural_vars)}")
print(f"  Volumes: {len(volume_vars)}")
print(f"  Variables climáticas: {len(climate_vars)}")
print(f"  Predictores macro: {len(predictor_vars)}")
print(f"  TOTAL features base: {len(df.columns) - 1}")

display(df.head())

✓ Dataset filtrado (>= 2000-01-01): (6731, 99)
  Período: 2000-01-03 → 2025-11-10
  Columnas: 99 (1 date + 98 features base)

Composición del dataset:
  Targets agrícolas: 15
  Volumes: 27
  Variables climáticas: 10
  Predictores macro: 46
  TOTAL features base: 98


,date,Baltic_Dry_Index,Brent_Crude,Cocoa,Coffee,Copper,Corn,Cotton,Crude_Oil,Ethanol,...,psd_united_states_Ending_Stocks,psd_united_states_Exports,psd_united_states_Stock_to_Use_Ratio,psd_argentina_Production,psd_argentina_Exports,psd_argentina_Stock_to_Use_Ratio,psd_china_Imports,psd_china_Crush,psd_china_Ending_Stocks,psd_china_Stock_to_Use_Ratio
0,2000-01-03,NaN,NaN,830.0,116.500000,NaN,NaN,51.070000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,1320.0,NaN,836.0,116.250000,NaN,NaN,50.730000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,1329.0,NaN,831.0,118.599998,NaN,NaN,51.560001,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,1351.0,NaN,841.0,116.849998,NaN,NaN,52.080002,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,1368.0,NaN,853.0,114.150002,NaN,NaN,53.959999,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

## FEATURE ENGINEERING FASE 1: Temporal Features

Extraemos componentes temporales de la columna `date`:

- **year:** Año
- **month:** Mes (1-12)
- **quarter:** Trimestre (1-4)
- **day_of_week:** Día de la semana (0=Monday, 6=Sunday)
- **day_of_year:** Día del año (1-365/366)
- **week_of_year:** Semana del año (1-52/53)

### Diagnóstico de Calidad: Variables Faltantes

Antes de aplicar lags, verificamos qué variables del período 2000+ tienen datos disponibles y cuáles tienen missing estructural.

In [54]:
# Análisis de missing values ANTES de feature engineering
print("=" * 80)
print("DIAGNÓSTICO DE CALIDAD - VARIABLES BASE (2000+)")
print("=" * 80)

# Obtener columnas base (sin date)
base_cols_pre = [c for c in df.columns if c != 'date']

print(f"\nDataset: {len(df):,} observaciones × {len(base_cols_pre)} variables")
print(f"Período: {df['date'].min().date()} → {df['date'].max().date()}")

# Calcular missing por variable
missing_stats = pd.DataFrame({
    'variable': base_cols_pre,
    'missing_count': [df[col].isna().sum() for col in base_cols_pre],
    'missing_pct': [(df[col].isna().sum() / len(df) * 100) for col in base_cols_pre],
    'available_count': [df[col].notna().sum() for col in base_cols_pre]
}).sort_values('missing_pct', ascending=False)

# Variables con >50% missing (CRÍTICO)
critical_missing = missing_stats[missing_stats['missing_pct'] > 50]
print(f"\n🔴 CRÍTICO - Variables con >50% missing ({len(critical_missing)}):")
if len(critical_missing) > 0:
    for idx, row in critical_missing.iterrows():
        print(f"  {row['variable']:45s}: {row['missing_count']:,} ({row['missing_pct']:.1f}%) - SOLO {row['available_count']:,} obs disponibles")
else:
    print("  Ninguna variable en rango crítico")

# Variables con 20-50% missing (ADVERTENCIA)
warning_missing = missing_stats[(missing_stats['missing_pct'] >= 20) & (missing_stats['missing_pct'] <= 50)]
print(f"\n⚠️  ADVERTENCIA - Variables con 20-50% missing ({len(warning_missing)}):")
if len(warning_missing) > 0:
    for idx, row in warning_missing.head(10).iterrows():
        print(f"  {row['variable']:45s}: {row['missing_count']:,} ({row['missing_pct']:.1f}%)")
else:
    print("  Ninguna variable en este rango")

# Variables completas o casi completas (<5% missing)
good_quality = missing_stats[missing_stats['missing_pct'] < 5]
print(f"\n✓ BUENA CALIDAD - Variables con <5% missing: {len(good_quality)} de {len(base_cols_pre)}")

# Resumen por tipo
print(f"\n\nResumen por tipo de variable:")
agricultural_vars = [c for c in base_cols_pre if any(ag in c for ag in AGRICULTURAL_COMMODITIES) and not c.endswith('_volume')]
volume_vars = [c for c in base_cols_pre if c.endswith('_volume')]
predictor_vars = [c for c in base_cols_pre if c not in agricultural_vars and c not in volume_vars]

for var_type, var_list in [('Targets agrícolas', agricultural_vars), 
                             ('Volumes', volume_vars), 
                             ('Predictores', predictor_vars)]:
    if var_list:
        type_missing = missing_stats[missing_stats['variable'].isin(var_list)]
        avg_missing = type_missing['missing_pct'].mean()
        max_missing = type_missing['missing_pct'].max()
        print(f"  {var_type:20s}: {len(var_list):2d} vars - Avg missing: {avg_missing:5.1f}% | Max: {max_missing:5.1f}%")

print("=" * 80)

DIAGNÓSTICO DE CALIDAD - VARIABLES BASE (2000+)

Dataset: 6,731 observaciones × 98 variables
Período: 2000-01-03 → 2025-11-10

🔴 CRÍTICO - Variables con >50% missing (0):
  Ninguna variable en rango crítico

⚠️  ADVERTENCIA - Variables con 20-50% missing (5):
  Brent_Crude_volume                           : 2,181 (32.4%)
  Brent_Crude                                  : 2,181 (32.4%)
  Ethanol_volume                               : 1,716 (25.5%)
  Ethanol                                      : 1,716 (25.5%)
  USD_BRL                                      : 1,456 (21.6%)

✓ BUENA CALIDAD - Variables con <5% missing: 47 de 98


Resumen por tipo de variable:
  Targets agrícolas   : 15 vars - Avg missing:   6.3% | Max:  15.0%
  Volumes             : 27 vars - Avg missing:   8.3% | Max:  32.4%
  Predictores         : 56 vars - Avg missing:   5.6% | Max:  32.4%

🔴 CRÍTICO - Variables con >50% missing (0):
  Ninguna variable en rango crítico

⚠️  ADVERTENCIA - Variables con 20-50% missing (5):


In [55]:
def add_temporal_features(df):
    """
    Agrega features temporales avanzadas extraídas de la columna date
    
    Incluye:
    - Componentes básicos: year, month, quarter, day_of_week, etc.
    - Indicadores de fin de período: is_month_end, is_quarter_end, is_year_end
    - Estacionalidad: season, is_harvest_season (Jun-Oct para granos USA)
    - Días transcurridos: days_since_year_start
    
    Args:
        df (pd.DataFrame): Dataset con columna 'date'
        
    Returns:
        pd.DataFrame: Dataset con features temporales
    """
    df = df.copy()
    
    # Componentes básicos
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['day_of_week'] = df['date'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week
    
    # Indicadores de fin de período (útiles para efectos de rebalanceo)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(int)
    df['is_year_end'] = df['date'].dt.is_year_end.astype(int)
    
    # Días desde inicio del año (captura tendencia intra-anual)
    df['days_since_year_start'] = (df['date'] - pd.to_datetime(df['year'].astype(str) + '-01-01')).dt.days
    
    # Estacionalidad: season (1=Winter, 2=Spring, 3=Summer, 4=Fall)
    # Northern Hemisphere seasons (USA grain production)
    df['season'] = df['month'].map({
        12: 1, 1: 1, 2: 1,  # Winter
        3: 2, 4: 2, 5: 2,   # Spring
        6: 3, 7: 3, 8: 3,   # Summer
        9: 4, 10: 4, 11: 4  # Fall
    })
    
    # Harvest season para granos USA: Junio-Octubre (peak en Jul-Sep)
    df['is_harvest_season'] = df['month'].isin([6, 7, 8, 9, 10]).astype(int)
    
    # Planting season para granos USA: Marzo-Mayo
    df['is_planting_season'] = df['month'].isin([3, 4, 5]).astype(int)
    
    print(f"✓ Temporal features agregadas: 13 columnas")
    print(f"  Básicas: year, month, quarter, day_of_week, day_of_year, week_of_year")
    print(f"  Indicadores: is_month_end, is_quarter_end, is_year_end")
    print(f"  Estacionalidad: season, is_harvest_season, is_planting_season, days_since_year_start")
    print(f"  Rango temporal: {df['year'].min()} → {df['year'].max()}")
    
    return df

# Aplicar transformación
df = add_temporal_features(df)

# Visualizar features temporales
temporal_cols = ['date', 'year', 'month', 'quarter', 'season', 
                 'is_harvest_season', 'is_planting_season', 'is_month_end']
display(df[temporal_cols].head(10))
display(df[temporal_cols].tail(10))

✓ Temporal features agregadas: 13 columnas
  Básicas: year, month, quarter, day_of_week, day_of_year, week_of_year
  Indicadores: is_month_end, is_quarter_end, is_year_end
  Estacionalidad: season, is_harvest_season, is_planting_season, days_since_year_start
  Rango temporal: 2000 → 2025


,date,year,month,quarter,season,is_harvest_season,is_planting_season,is_month_end
0,2000-01-03,2000,1,1,1,0,0,0
1,2000-01-04,2000,1,1,1,0,0,0
2,2000-01-05,2000,1,1,1,0,0,0
3,2000-01-06,2000,1,1,1,0,0,0
4,2000-01-07,2000,1,1,1,0,0,0
5,2000-01-10,2000,1,1,1,0,0,0
6,2000-01-11,2000,1,1,1,0,0,0
7,2000-01-12,2000,1,1,1,0,0,0
8,2000-01-13,2000,1,1,1,0,0,0
9,2000-01-14,2000,1,1,1,0,0,0


,date,year,month,quarter,season,is_harvest_season,is_planting_season,is_month_end
6721,2025-10-28,2025,10,4,4,1,0,0
6722,2025-10-29,2025,10,4,4,1,0,0
6723,2025-10-30,2025,10,4,4,1,0,0
6724,2025-10-31,2025,10,4,4,1,0,1
6725,2025-11-03,2025,11,4,4,0,0,0
6726,2025-11-04,2025,11,4,4,0,0,0
6727,2025-11-05,2025,11,4,4,0,0,0
6728,2025-11-06,2025,11,4,4,0,0,0
6729,2025-11-07,2025,11,4,4,0,0,0
6730,2025-11-10,2025,11,4,4,0,0,0


### Verificación de Temporal Features

Verificamos distribución de features temporales:

In [56]:
# Distribución por año
print("Observaciones por año:")
year_counts = df['year'].value_counts().sort_index()
print(year_counts.to_string())

# Distribución por mes (verificar balance)
print("\n\nObservaciones por mes:")
month_counts = df['month'].value_counts().sort_index()
print(month_counts.to_string())

# Distribución por día de la semana (detectar sesgo de trading days)
print("\n\nObservaciones por día de la semana:")
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
day_counts = df['day_of_week'].value_counts().sort_index()
for day_num, count in day_counts.items():
    print(f"  {day_names[day_num]:12s}: {count:,}")

# Verificar estacionalidad agrícola
print("\n\nEstacionalidad agrícola:")
print(f"  Observaciones en harvest season (Jun-Oct): {df['is_harvest_season'].sum():,} ({df['is_harvest_season'].mean()*100:.1f}%)")
print(f"  Observaciones en planting season (Mar-May): {df['is_planting_season'].sum():,} ({df['is_planting_season'].mean()*100:.1f}%)")

# Verificar distribución por season
print("\n\nObservaciones por season:")
season_names = {1: 'Winter (Dec-Feb)', 2: 'Spring (Mar-May)', 3: 'Summer (Jun-Aug)', 4: 'Fall (Sep-Nov)'}
season_counts = df['season'].value_counts().sort_index()
for season_num, count in season_counts.items():
    print(f"  {season_names[season_num]:25s}: {count:,}")

# Verificar indicadores de fin de período
print("\n\nIndicadores de fin de período:")
print(f"  Month-end days: {df['is_month_end'].sum():,}")
print(f"  Quarter-end days: {df['is_quarter_end'].sum():,}")
print(f"  Year-end days: {df['is_year_end'].sum():,}")

Observaciones por año:
year
2000    257
2001    257
2002    258
2003    258
2004    262
2005    260
2006    260
2007    261
2008    262
2009    261
2010    261
2011    260
2012    261
2013    261
2014    261
2015    261
2016    261
2017    260
2018    261
2019    261
2020    262
2021    261
2022    260
2023    260
2024    262
2025    222


Observaciones por mes:
month
1     573
2     525
3     574
4     552
5     575
6     556
7     575
8     577
9     556
10    577
11    543
12    548


Observaciones por día de la semana:
  Monday      : 1,345
  Tuesday     : 1,347
  Wednesday   : 1,346
  Thursday    : 1,349
  Friday      : 1,344


Estacionalidad agrícola:
  Observaciones en harvest season (Jun-Oct): 2,841 (42.2%)
  Observaciones en planting season (Mar-May): 1,701 (25.3%)


Observaciones por season:
  Winter (Dec-Feb)         : 1,646
  Spring (Mar-May)         : 1,701
  Summer (Jun-Aug)         : 1,708
  Fall (Sep-Nov)           : 1,676


Indicadores de fin de período:
  Month-end da

---

## FEATURE ENGINEERING FASE 2: Lag Features

Creamos variables rezagadas (lags) para capturar **memoria temporal** en las series:

- **Lag 1:** Valor del día anterior (t-1)
- **Lag 7:** Valor de hace 1 semana (t-7)
- **Lag 30:** Valor de hace 1 mes (t-30)

**Aplicamos lags a:**
- Todos los precios de commodities
- Todos los predictores macro
- Volúmenes de transacciones

**Total esperado:** ~50 columnas base × 3 lags = **~150 features de lag**

---

## ANÁLISIS PREVIO: Estrategia de Lags Diferenciada

### Justificación Metodológica

El análisis exploratorio (Notebook 1.2 - Correlaciones) reveló patrones de asociación heterogéneos entre predictores y precios agrícolas. Esto sugiere que **no todas las variables requieren la misma profundidad histórica** (lag depth) para capturar su relación con los targets.

### Clasificación de Variables según Correlación

Según `correlation_predictores_agricolas.csv`, identificamos 3 grupos de predictores:

**1. Predictores de Alta Correlación (|r| > 0.70):**
- **Energía:** Crude_Oil (0.74-0.81), Heating_Oil (0.81-0.84), RBOB_Gasoline (0.79-0.82), Brent_Crude (0.71-0.73)
- **Metales:** Copper (0.74-0.81), Gold (0.58-0.86), Silver (0.63-0.84)
- **Índices:** Energy_Index (0.59-0.77), Materials_Index (0.37-0.82)

**2. Predictores de Correlación Moderada (0.40 < |r| < 0.70):**
- **Macro:** SP500, TIPS, Treasury_10Y
- **FX:** USD_ARS, USD_BRL, USD_RUB, USD_UAH
- **Clima:** Temp_Global_Grain, GDD_Global_Grain, ET0_Global_Grain

**3. Predictores Estructurales (frecuencia mensual/trimestral):**
- **Supply-Demand:** Variables PSD (Production, Ending_Stocks, Exports, Imports, Stock_to_Use_Ratio)
- **Clima Acumulado:** Heat_Stress_Days, Precip_Deficit

### Estrategia de Lags Implementada

**Targets (precios agrícolas):**  
`[1, 2, 3, 5, 7]` - Lags cortos para capturar momentum reciente. Necesarios para predecir t+1.

**Predictores alta correlación:**  
`[1, 7, 14, 30]` - Lags medianos para capturar transmisión de shocks (ej: petróleo → costos de transporte → precios).

**Predictores correlación moderada:**  
`[1, 7, 30]` - Lags selectivos para balance entre información y parsimonia.

**Predictores estructurales (PSD, clima acumulado):**  
`[30, 60, 90]` - Lags largos porque se actualizan mensual/trimestralmente. Información de inventarios tiene efecto prolongado.

### Trade-off: Historia vs Observaciones

- **Lag máximo = 90 días:** perdemos ~3 meses de observaciones al inicio (90/6731 = 1.3% del dataset)
- **Beneficio:** capturamos ciclos estacionales (trimestres) y efectos acumulados (stocks, clima)
- **Costo computacional:** ~50 variables base × promedio 4 lags = **~200 features de lag**

In [57]:
def add_lag_features_differentiated(df, agricultural_commodities):
    """
    Agrega features de lags con estrategia diferenciada según tipo de variable
    
    Args:
        df (pd.DataFrame): Dataset base con columna 'date'
        agricultural_commodities (list): Lista de commodities agrícolas (targets)
        
    Returns:
        pd.DataFrame: Dataset con lags diferenciados
    """
    df = df.copy()
    
    # Identificar tipos de variables
    temporal_cols = ['year', 'month', 'quarter', 'day_of_week', 'day_of_year', 'week_of_year']
    all_cols = [c for c in df.columns if c not in ['date'] + temporal_cols]
    
    # 1. Targets: commodities agrícolas (lags cortos)
    target_cols = [c for c in all_cols if any(ag in c for ag in agricultural_commodities)]
    target_price_cols = [c for c in target_cols if not c.endswith('_volume')]
    
    # 2. Predictores de alta correlación (energía, metales)
    high_corr_predictors = [
        'Crude_Oil', 'Heating_Oil', 'RBOB_Gasoline', 'Brent_Crude',
        'Copper', 'Gold', 'Silver', 'Palladium', 'Platinum',
        'Energy_Index', 'Materials_Index'
    ]
    high_corr_cols = [c for c in all_cols if any(p in c for p in high_corr_predictors)]
    
    # 3. Predictores estructurales (PSD + clima acumulado)
    structural_predictors = ['psd_', 'Heat_Stress_Days', 'Precip_Deficit']
    structural_cols = [c for c in all_cols if any(p in c for p in structural_predictors)]
    
    # 4. Resto de predictores (correlación moderada)
    moderate_cols = [c for c in all_cols 
                     if c not in target_cols 
                     and c not in high_corr_cols 
                     and c not in structural_cols]
    
    features_added = 0
    
    # Aplicar lags diferenciados
    print("Aplicando lags diferenciados...")
    
    # Targets: lags [1,2,3,5,7]
    for col in target_price_cols:
        if col in df.columns:
            for lag in [1, 2, 3, 5, 7]:
                df[f'{col}_lag{lag}'] = df[col].shift(lag)
                features_added += 1
    print(f"  ✓ Targets (agricultura): {len(target_price_cols)} vars × 5 lags = {len(target_price_cols)*5} features")
    
    # Predictores alta correlación: lags [1,7,14,30]
    for col in high_corr_cols:
        if col in df.columns:
            for lag in [1, 7, 14, 30]:
                df[f'{col}_lag{lag}'] = df[col].shift(lag)
                features_added += 1
    print(f"  ✓ Alta correlación (energía/metales): {len(high_corr_cols)} vars × 4 lags = {len(high_corr_cols)*4} features")
    
    # Predictores estructurales: lags [30,60,90]
    for col in structural_cols:
        if col in df.columns:
            for lag in [30, 60, 90]:
                df[f'{col}_lag{lag}'] = df[col].shift(lag)
                features_added += 1
    print(f"  ✓ Estructurales (PSD/clima): {len(structural_cols)} vars × 3 lags = {len(structural_cols)*3} features")
    
    # Predictores moderados: lags [1,7,30]
    for col in moderate_cols:
        if col in df.columns:
            for lag in [1, 7, 30]:
                df[f'{col}_lag{lag}'] = df[col].shift(lag)
                features_added += 1
    print(f"  ✓ Moderados (macro/FX/clima): {len(moderate_cols)} vars × 3 lags = {len(moderate_cols)*3} features")
    
    print(f"\n✓ Total lag features: {features_added}")
    return df

# Aplicar lags diferenciados
df = add_lag_features_differentiated(df, AGRICULTURAL_COMMODITIES)

Aplicando lags diferenciados...
  ✓ Targets (agricultura): 15 vars × 5 lags = 75 features
  ✓ Alta correlación (energía/metales): 20 vars × 4 lags = 80 features
  ✓ Alta correlación (energía/metales): 20 vars × 4 lags = 80 features
  ✓ Estructurales (PSD/clima): 22 vars × 3 lags = 66 features
  ✓ Moderados (macro/FX/clima): 33 vars × 3 lags = 99 features

✓ Total lag features: 320
  ✓ Estructurales (PSD/clima): 22 vars × 3 lags = 66 features
  ✓ Moderados (macro/FX/clima): 33 vars × 3 lags = 99 features

✓ Total lag features: 320


C:\Users\AdministradorIT\AppData\Local\Temp\ipykernel_15320\77954874.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_lag{lag}'] = df[col].shift(lag)
C:\Users\AdministradorIT\AppData\Local\Temp\ipykernel_15320\77954874.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_lag{lag}'] = df[col].shift(lag)
C:\Users\AdministradorIT\AppData\Local\Temp\ipykernel_15320\77954874.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has 

### Verificación de Lag Features

In [58]:
# Verificación de lag features

# 1. Ejemplo: visualizar lags para Corn (target agrícola)
if 'Corn' in df.columns:
    corn_cols = ['date', 'Corn', 'Corn_lag1', 'Corn_lag2', 'Corn_lag3', 'Corn_lag5', 'Corn_lag7']
    print("Ejemplo - Lags de Corn (target agrícola):")
    display(df[corn_cols].iloc[7:17])
    
    # Verificar que lag1 = valor de t-1
    sample_idx = 10
    print(f"\nVerificación lag1 (índice {sample_idx}):")
    print(f"  Corn[{sample_idx}] = {df.loc[sample_idx, 'Corn']:.2f}")
    print(f"  Corn_lag1[{sample_idx+1}] = {df.loc[sample_idx+1, 'Corn_lag1']:.2f}")
    print(f"  ¿Coinciden? {df.loc[sample_idx, 'Corn'] == df.loc[sample_idx+1, 'Corn_lag1']}")

# 2. Ejemplo: visualizar lags para Crude_Oil (predictor alta correlación)
if 'Crude_Oil' in df.columns:
    oil_cols = ['date', 'Crude_Oil', 'Crude_Oil_lag1', 'Crude_Oil_lag7', 'Crude_Oil_lag14', 'Crude_Oil_lag30']
    print("\n\nEjemplo - Lags de Crude_Oil (predictor alta correlación):")
    display(df[oil_cols].iloc[30:40])

# 3. Contar missing values introducidos por lags
lag_cols = [col for col in df.columns if '_lag' in col]
print(f"\n\nMissing values introducidos por lags:")
print(f"  Total lag features: {len(lag_cols)}")

# Agrupar por tipo de lag
lag_types = {}
for col in lag_cols:
    lag_num = int(col.split('_lag')[1])
    if lag_num not in lag_types:
        lag_types[lag_num] = []
    lag_types[lag_num].append(col)

for lag_num in sorted(lag_types.keys()):
    sample_col = lag_types[lag_num][0]
    n_missing = df[sample_col].isna().sum()
    print(f"  Lag{lag_num}: {n_missing} NaN (primeras {lag_num} observaciones)")
    print(f"          Aplicado a {len(lag_types[lag_num])} variables")

# 4. Observaciones perdidas por lag máximo
max_lag = max(lag_types.keys())
obs_lost = df[[col for col in lag_cols if f'_lag{max_lag}' in col][0]].isna().sum()
pct_lost = (obs_lost / len(df)) * 100
print(f"\nObservaciones perdidas por lag máximo ({max_lag} días):")
print(f"  {obs_lost} / {len(df)} = {pct_lost:.2f}% del dataset")

Ejemplo - Lags de Corn (target agrícola):


,date,Corn,Corn_lag1,Corn_lag2,Corn_lag3,Corn_lag5,Corn_lag7
7,2000-01-12,NaN,NaN,NaN,NaN,NaN,NaN
8,2000-01-13,NaN,NaN,NaN,NaN,NaN,NaN
9,2000-01-14,NaN,NaN,NaN,NaN,NaN,NaN
10,2000-01-17,NaN,NaN,NaN,NaN,NaN,NaN
11,2000-01-18,NaN,NaN,NaN,NaN,NaN,NaN
12,2000-01-19,NaN,NaN,NaN,NaN,NaN,NaN
13,2000-01-20,NaN,NaN,NaN,NaN,NaN,NaN
14,2000-01-21,NaN,NaN,NaN,NaN,NaN,NaN
15,2000-01-24,NaN,NaN,NaN,NaN,NaN,NaN
16,2000-01-25,NaN,NaN,NaN,NaN,NaN,NaN



Verificación lag1 (índice 10):
  Corn[10] = nan
  Corn_lag1[11] = nan
  ¿Coinciden? False


Ejemplo - Lags de Crude_Oil (predictor alta correlación):


,date,Crude_Oil,Crude_Oil_lag1,Crude_Oil_lag7,Crude_Oil_lag14,Crude_Oil_lag30
30,2000-02-14,NaN,NaN,NaN,NaN,NaN
31,2000-02-15,NaN,NaN,NaN,NaN,NaN
32,2000-02-16,NaN,NaN,NaN,NaN,NaN
33,2000-02-17,NaN,NaN,NaN,NaN,NaN
34,2000-02-18,NaN,NaN,NaN,NaN,NaN
35,2000-02-21,NaN,NaN,NaN,NaN,NaN
36,2000-02-22,NaN,NaN,NaN,NaN,NaN
37,2000-02-23,NaN,NaN,NaN,NaN,NaN
38,2000-02-24,NaN,NaN,NaN,NaN,NaN
39,2000-02-25,NaN,NaN,NaN,NaN,NaN




Missing values introducidos por lags:
  Total lag features: 320
  Lag1: 247 NaN (primeras 1 observaciones)
          Aplicado a 68 variables
  Lag2: 248 NaN (primeras 2 observaciones)
          Aplicado a 15 variables
  Lag3: 249 NaN (primeras 3 observaciones)
          Aplicado a 15 variables
  Lag5: 251 NaN (primeras 5 observaciones)
          Aplicado a 15 variables
  Lag7: 253 NaN (primeras 7 observaciones)
          Aplicado a 68 variables
  Lag14: 2195 NaN (primeras 14 observaciones)
          Aplicado a 20 variables
  Lag30: 2211 NaN (primeras 30 observaciones)
          Aplicado a 75 variables
  Lag60: 60 NaN (primeras 60 observaciones)
          Aplicado a 22 variables
  Lag90: 90 NaN (primeras 90 observaciones)
          Aplicado a 22 variables

Observaciones perdidas por lag máximo (90 días):
  90 / 6731 = 1.34% del dataset


---

## 5. Resumen del Dataset con Temporal & Lag Features

In [59]:
print("=" * 80)
print("RESUMEN - DATASET CON TEMPORAL & LAG FEATURES DIFERENCIADOS")
print("=" * 80)

print(f"\nDimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Período: {df['date'].min().date()} → {df['date'].max().date()}")

# Contar tipos de features
temporal_cols = ['year', 'month', 'quarter', 'day_of_week', 'day_of_year', 'week_of_year',
                 'is_month_end', 'is_quarter_end', 'is_year_end', 'days_since_year_start',
                 'season', 'is_harvest_season', 'is_planting_season']
lag_cols = [c for c in df.columns if '_lag' in c]
volume_cols = [c for c in df.columns if '_volume' in c and '_lag' not in c]

# Contar base features (sin lags ni temporales)
base_cols = [c for c in df.columns 
             if c not in temporal_cols 
             and c not in lag_cols 
             and c != 'date']

print(f"\nFeatures por tipo:")
print(f"  Base (precios + predictores + volumes): {len(base_cols)}")
print(f"    - Precios agrícolas (targets): {len([c for c in base_cols if any(ag in c for ag in AGRICULTURAL_COMMODITIES) and not c.endswith('_volume')])}")
print(f"    - Volumes: {len(volume_cols)}")
print(f"    - Predictores: {len(base_cols) - len(volume_cols) - len([c for c in base_cols if any(ag in c for ag in AGRICULTURAL_COMMODITIES)])}")
print(f"  Temporales: {len(temporal_cols)}")
print(f"  Lags: {len(lag_cols)}")
print(f"  TOTAL: {1 + len(base_cols) + len(temporal_cols) + len(lag_cols)} (incluyendo date)")

# Desglose de lags por tipo
print(f"\nDesglose de lag features por estrategia:")
lag_short = len([c for c in lag_cols if any(f'_lag{i}' in c for i in [1,2,3,5,7])])
lag_medium = len([c for c in lag_cols if any(f'_lag{i}' in c for i in [14,30])])
lag_long = len([c for c in lag_cols if any(f'_lag{i}' in c for i in [60,90])])
print(f"  Lags cortos [1,2,3,5,7]: ~{lag_short} (targets agrícolas)")
print(f"  Lags medianos [14,30]: ~{lag_medium} (predictores correlacionados)")
print(f"  Lags largos [60,90]: ~{lag_long} (predictores estructurales)")

# Missing values
print(f"\nMissing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100)

# Separar missing de lags vs missing estructural
lag_missing = missing[lag_cols].sum()
base_missing = missing[[c for c in base_cols if c in df.columns]].sum()

print(f"  Missing en base features: {base_missing:,} ({base_missing/df[base_cols].size*100:.2f}%)")
print(f"  Missing por lags (esperado): {lag_missing:,}")
print(f"  Missing total: {missing.sum():,} ({missing.sum()/df.size*100:.2f}%)")

# Top missing por variable base (excluir lags)
print(f"\nTop 10 variables base con más missing:")
top_missing = missing_pct[[c for c in base_cols if c in df.columns]].nlargest(10)
for col, pct in top_missing.items():
    if pct > 0:
        print(f"  {col:45s}: {missing[col]:6,} ({pct:5.2f}%)")

print("=" * 80)

RESUMEN - DATASET CON TEMPORAL & LAG FEATURES DIFERENCIADOS

Dimensiones: 6,731 filas × 432 columnas
Período: 2000-01-03 → 2025-11-10

Features por tipo:
  Base (precios + predictores + volumes): 98
    - Precios agrícolas (targets): 15
    - Volumes: 27
    - Predictores: 41
  Temporales: 13
  Lags: 320
  TOTAL: 432 (incluyendo date)

Desglose de lag features por estrategia:
  Lags cortos [1,2,3,5,7]: ~276 (targets agrícolas)
  Lags medianos [14,30]: ~95 (predictores correlacionados)
  Lags largos [60,90]: ~44 (predictores estructurales)

Missing values:
  Missing en base features: 42,484 (6.44%)
  Missing por lags (esperado): 140,450
  Missing total: 182,934 (6.29%)

Top 10 variables base con más missing:
  Brent_Crude                                  :  2,181 (32.40%)
  Brent_Crude_volume                           :  2,181 (32.40%)
  Ethanol                                      :  1,716 (25.49%)
  Ethanol_volume                               :  1,716 (25.49%)
  USD_BRL              

---

## 6. Guardar Dataset Intermedio

Guardamos el dataset con temporal y lag features para usar en los siguientes notebooks de feature engineering.

In [60]:
# Guardar dataset intermedio
output_file = PROCESSED_DIR / 'features_step1_temporal_lags.csv'
df.to_csv(output_file, index=False)

file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"✓ Dataset guardado: {output_file.name}")
print(f"  Tamaño: {file_size_mb:.2f} MB")
print(f"  Dimensiones: {df.shape}")

# Crear metadata JSON con información de las features
temporal_cols = ['year', 'month', 'quarter', 'day_of_week', 'day_of_year', 'week_of_year',
                 'is_month_end', 'is_quarter_end', 'is_year_end', 'days_since_year_start',
                 'season', 'is_harvest_season', 'is_planting_season']
lag_cols = [c for c in df.columns if '_lag' in c]
base_cols = [c for c in df.columns 
             if c not in temporal_cols 
             and c not in lag_cols 
             and c != 'date']

metadata_features = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'archivo_output': output_file.name,
    'dataset': {
        'observaciones': int(len(df)),
        'columnas_totales': int(len(df.columns)),
        'periodo': f"{df['date'].min().date()} - {df['date'].max().date()}"
    },
    'features': {
        'base': {
            'total': int(len(base_cols)),
            'targets_agricolas': int(len([c for c in base_cols if any(ag in c for ag in AGRICULTURAL_COMMODITIES) and not c.endswith('_volume')])),
            'volumes': int(len([c for c in base_cols if c.endswith('_volume')])),
            'predictores': int(len(base_cols) - len([c for c in base_cols if any(ag in c for ag in AGRICULTURAL_COMMODITIES)]))
        },
        'temporales': {
            'total': int(len(temporal_cols)),
            'columnas': temporal_cols
        },
        'lags': {
            'total': int(len(lag_cols)),
            'estrategia': {
                'targets_agricolas': 'lags [1,2,3,5,7] - capturar momentum reciente',
                'predictores_alta_correlacion': 'lags [1,7,14,30] - transmisión de shocks',
                'predictores_moderados': 'lags [1,7,30] - balance información/parsimonia',
                'predictores_estructurales': 'lags [30,60,90] - frecuencia mensual/trimestral'
            },
            'observaciones_perdidas_lag_max': int(df[lag_cols[0]].isna().sum())
        }
    },
    'missing_values': {
        'total': int(df.isnull().sum().sum()),
        'porcentaje_global': float(df.isnull().sum().sum() / df.size * 100),
        'por_lags': int(df[lag_cols].isnull().sum().sum()),
        'por_base_features': int(df[base_cols].isnull().sum().sum())
    }
}

metadata_file = PROCESSED_DIR / 'metadata_features_step1.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata_features, f, indent=2)

print(f"\n✓ Metadata exportado: {metadata_file.name}")

✓ Dataset guardado: features_step1_temporal_lags.csv
  Tamaño: 27.86 MB
  Dimensiones: (6731, 432)

✓ Metadata exportado: metadata_features_step1.json
